#Mount Drive + Start Spark

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install pyspark kafka-python

from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("Yelp_Checkin_Kafka_Pipeline")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

print("Spark ready:", spark.version)


#Install & Start Kafka in Colab (Zookeeper + Broker) + Create Topic

In [ ]:
# Java (required)
!apt-get -qq update
!apt-get -qq install -y default-jre

# Download Kafka
!wget -q https://downloads.apache.org/kafka/3.7.0/kafka_2.13-3.7.0.tgz
!tar -xzf kafka_2.13-3.7.0.tgz

# Start Zookeeper and Kafka broker
!nohup kafka_2.13-3.7.0/bin/zookeeper-server-start.sh kafka_2.13-3.7.0/config/zookeeper.properties > zookeeper.log 2>&1 &
!sleep 5
!nohup kafka_2.13-3.7.0/bin/kafka-server-start.sh kafka_2.13-3.7.0/config/server.properties > kafka.log 2>&1 &
!sleep 8

# Create the topic
!kafka_2.13-3.7.0/bin/kafka-topics.sh --create \
  --topic checkin_stream --bootstrap-server localhost:9092 \
  --partitions 1 --replication-factor 1

!kafka_2.13-3.7.0/bin/kafka-topics.sh --list --bootstrap-server localhost:9092


#Build “Check-in Event” dataset from Yelp checkin CSV (event-level)

In [ ]:
from pyspark.sql import functions as F

BASE = "/content/drive/MyDrive/CourseWork/CSV_Files"
CHECKIN_PATH = f"{BASE}/yelp_academic_dataset_checkin.csv"

df_checkin = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv(CHECKIN_PATH))

# Explode comma-separated timestamp strings into event rows (business_id, checkin_ts)
df_events = (df_checkin
    .select("business_id", F.explode(F.split(F.col("date"), r",\s*")).alias("checkin_ts_raw"))
    .withColumn("checkin_ts", F.to_timestamp("checkin_ts_raw"))
    .drop("checkin_ts_raw")
    .dropna(subset=["business_id", "checkin_ts"])
)

print("Event rows (sample):")
df_events.show(5, truncate=False)


#Python Producer (publishes “Check-in Event” JSON to Kafka topic)

In [ ]:
import json, time
from kafka import KafkaProducer

# Keep the demo light (you can increase if needed)
pdf_events = df_events.select("business_id", "checkin_ts").limit(2000).toPandas()
pdf_events["checkin_ts"] = pdf_events["checkin_ts"].astype(str)

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

topic = "checkin_stream"

for _, r in pdf_events.iterrows():
    event = {"business_id": r["business_id"], "checkin_ts": r["checkin_ts"]}
    producer.send(topic, event)
    time.sleep(0.01)  # simulate live check-ins (10ms/event)

producer.flush()
print(f"Published {len(pdf_events)} check-in events to topic: {topic}")


#PySpark Structured Streaming Consumer (subscribes to Kafka topic)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("business_id", StringType(), True),
    StructField("checkin_ts",  StringType(), True)
])

kafka_stream = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "checkin_stream")
    .option("startingOffsets", "earliest")
    .load()
)

events = (kafka_stream
    .selectExpr("CAST(value AS STRING) AS json_str")
    .select(F.from_json("json_str", schema).alias("x"))
    .select("x.*")
    .withColumn("checkin_ts", F.to_timestamp("checkin_ts"))
    .dropna(subset=["business_id", "checkin_ts"])
    .withColumn("weekday", F.date_format("checkin_ts", "E"))
    .withColumn("hour", F.hour("checkin_ts"))
)


#Streaming Analysis

## 1. Windowed Aggregation (check-ins per 1-minute window)

In [ ]:
windowed_agg = (events
    .withWatermark("checkin_ts", "2 minutes")
    .groupBy(F.window("checkin_ts", "1 minute").alias("w"))
    .agg(F.count("*").alias("checkins_1min"))
    .select(F.col("w.start").alias("window_start"),
            F.col("w.end").alias("window_end"),
            "checkins_1min")
)

##2. Weekday × Hour Intensity (heatmap-ready counts)

In [ ]:
weekday_hour_intensity = (events
    .groupBy("weekday", "hour")
    .agg(F.count("*").alias("checkins"))
)

##3. High-Traffic Flag (threshold logic)

In [ ]:
THRESHOLD = 30

high_traffic_flag = (windowed_agg
    .withColumn("high_traffic",
                F.when(F.col("checkins_1min") >= F.lit(THRESHOLD), F.lit(1)).otherwise(F.lit(0)))
)

#Processed Footfall Metrics” (merge outputs) + Save to Drive

##Write the 3 outputs to Drive (Parquet) for Tableau

In [ ]:
OUT_BASE = "/content/drive/MyDrive/CourseWork/Streaming_Output"
CHK_BASE = "/content/drive/MyDrive/CourseWork/Streaming_Output/_checkpoints"

# Windowed + High traffic
q1 = (high_traffic_flag.writeStream
      .format("parquet")
      .outputMode("append")
      .option("path", f"{OUT_BASE}/windowed_high_traffic")
      .option("checkpointLocation", f"{CHK_BASE}/windowed_high_traffic")
      .start())

# Weekday × Hour intensity (heatmap-ready)
q2 = (weekday_hour_intensity.writeStream
      .format("parquet")
      .outputMode("complete")
      .option("path", f"{OUT_BASE}/weekday_hour_intensity")
      .option("checkpointLocation", f"{CHK_BASE}/weekday_hour_intensity")
      .start())

print("Streaming queries started. Let them run ~30–60 seconds, then stop.")


##Stopping when done

In [ ]:
# Stop streams when obtained enough output
q1.stop()
q2.stop()
print("Streams stopped.")

#Console preview during demo

In [ ]:
# Console preview for windowed metrics
preview = (high_traffic_flag.writeStream
           .format("console")
           .outputMode("append")
           .option("truncate", "false")
           .start())